<a href="https://colab.research.google.com/github/maierav/ai_oscp_neuro/blob/main/notebooks/sensorimotor_mismatch_ecephys.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Sensorimotor mismatch — motor-based prediction error (Neuropixels)

The other analyses in this repo test **sensory** prediction error (an oddball orientation, an
omitted grating). The sensorimotor paradigm tests a different prediction: the brain's model of
**how its own movement should change what it sees**. Visual flow is coupled to the animal's
running on a wheel (closed loop). A `motor_halt` freezes the flow mid-run, a `motor_omission`
drops it, and orientation deviants rotate it — each violating the motor–sensory contingency.

**The designed control** is `Control block 4` (`open_loop_prerecorded`): the *identical* visual
events played back **decoupled** from running. A halt you did not cause is not a prediction
violation, so *closed-loop − open-loop* isolates the motor-based error.

**The catch, which this notebook makes explicit:** a motor–sensory mismatch only exists **while
the animal is running**, but these mice are stationary most of the time, and the open-loop control
block contains only **8 events per type**. Running has to coincide with those 8 events — and across
8 CCF sessions it does so cleanly in only one (sub-830794). So we report **two** contrasts:

1. **Within-block deviance** (all 8 sessions, well-powered): does each deviant exceed the ongoing
   standard flow? Tests deviance detection in the sensorimotor block.
2. **Closed-loop vs open-loop** (single powered session + motor-dependence of the halt): does the
   response depend on the *motor* contingency, not just the visual event?

In [ ]:
# Setup
import sys, subprocess
def _pip(*p): subprocess.run([sys.executable,"-m","pip","install","-q",*p],check=True)
try:
    import pynwb, remfile, h5py, fsspec, aiohttp  # noqa
except ImportError:
    _pip("pynwb","remfile","h5py","fsspec","aiohttp","requests")
import numpy as np, pandas as pd, h5py, remfile, requests, re, pickle, time
from scipy import stats as ss
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt

QUICK = True   # True: 3 sessions incl. the one powered session (830794), fast Colab pass. False: all 8.

In [ ]:
# NWB streaming + CCF-corrected unit->area mapping
def s3_url(ds, aid, version="draft"):
    u=f"https://api.dandiarchive.org/api/dandisets/{ds}/versions/{version}/assets/{aid}/download/"
    return requests.get(u,allow_redirects=False,timeout=60).headers["Location"]
def col(g,c):
    v=g[c][:]; return np.array([x.decode() if isinstance(x,bytes) else x for x in v])
def get_running(fh):
    rs=fh["processing"]["running"]["running_speed"]; return rs["timestamps"][:],rs["data"][:]
def decode_layer(acr):
    m=re.match(r"(VIS[a-z]*)(\d[a-b]?)?",acr)
    return (acr,"") if not m else (m.group(1),(m.group(2) or ""))

# 8 CCF sensorimotor sessions (subject, DANDI 001637 asset id)
SESSIONS=[("820459","e3cbee16-232a-4874-8b7e-cfbaebdfeeaf"),
          ("830849","ab8ec755-2b87-45c3-a29d-dd88955334b8"),
          ("830852","e15084a4-6372-4a2a-90a7-64ef0e4450b0"),
          ("830848","c53ec231-46f7-42d6-b89a-19e260b8eaa9"),
          ("830846","f2b15614-c07b-4187-9ace-f73b6e55005e"),
          ("834691","0cc012fd-e92c-4087-95ef-150d00d59b42"),
          ("830851","fc14f27c-4b7d-4332-8562-a4c824f757f3"),
          ("830794","0c67a84f-7da3-44b2-b707-72ab376a9034")]
if QUICK:
    SESSIONS=[s for s in SESSIONS if s[0] in ("830849","830846","830794")]
print(f"{len(SESSIONS)} sessions")

In [ ]:
# Per-session extractor: response scalars + PSTHs, split by running state and loop condition
PRE,POST,BW=0.5,1.0,0.075
EDGES=np.arange(-PRE,POST+BW,BW); TCEN=EDGES[:-1]+BW/2
RESP=(0.0,0.3); BASE=(-0.2,-0.01)          # motor-mismatch response / baseline windows
RUN_HI, REST_LO = 1.0, 0.5                 # cm/s thresholds (pre-event 1 s window)
EVENTS=["motor_halt","motor_omission","motor_orientation_45","motor_orientation_90"]

def sm_extract(subj,aid):
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    try:
        I=fh["intervals"]; gM=I["Sensory-motor mismatch block_presentations"]; gO=I["Control block 4_presentations"]
        rt,rv=get_running(fh)
        def prewin(times,win=(-1.0,0.0)):
            out=[]
            for t0 in times:
                m=(rt>=t0+win[0])&(rt<t0+win[1]); out.append(np.nanmean(rv[m]) if m.sum() else np.nan)
            return np.array(out)
        TTm=col(gM,"TrialType"); tsm=gM["start_time"][:]; TTo=col(gO,"TrialType"); tso=gO["start_time"][:]
        U=fh["units"]; st=U["spike_times"][:]; sti=U["spike_times_index"][:]
        n=len(sti); qc=U["default_qc"][:] if "default_qc" in U else np.ones(n,bool)
        Eg=fh["general/extracellular_ephys/electrodes"]; eloc=col(Eg,"location"); egroup=col(Eg,"group_name")
        dev=col(U,"device_name"); eci=U["extremum_channel_index"][:]
        groups=sorted(set(egroup),key=lambda gn:np.where(egroup==gn)[0][0])
        offs={gn:int(np.where(egroup==gn)[0][0]) for gn in groups}; bl={gn:int((egroup==gn).sum()) for gn in groups}
        def d2g(dd):
            for gn in groups:
                if dd[-1].lower() in gn.lower(): return gn
            return groups[0]
        dmap={dd:d2g(dd) for dd in set(dev)}
        # extremum_channel_index is PER-PROBE; offset into the stacked electrode table, clip to block length
        u_loc=np.array([eloc[offs[dmap[dev[i]]]+min(int(eci[i]),bl[dmap[dev[i]]]-1)] for i in range(n)])
        vis=np.where(qc & np.array([bool(re.match("VIS",r)) for r in u_loc]))[0]
        def sp_(i): return st[(0 if i==0 else sti[i-1]):sti[i]]
        def wrate(sp,times,win): return (np.searchsorted(sp,times+win[1])-np.searchsorted(sp,times+win[0]))/(win[1]-win[0])
        def psth(sp,times):
            if len(times)==0: return np.full(len(TCEN),np.nan)
            lo=np.searchsorted(sp,times.min()-PRE); hi=np.searchsorted(sp,times.max()+POST); sp2=sp[lo:hi]
            rel=(sp2[None,:]-times[:,None]).ravel(); rel=rel[(rel>=-PRE)&(rel<POST)]
            return np.histogram(rel,EDGES)[0]/(len(times)*BW)
        sM={e:prewin(tsm[TTm==e]) for e in EVENTS}; sO={e:prewin(tso[TTo==e]) for e in EVENTS}
        tM={e:tsm[TTm==e] for e in EVENTS}; tO={e:tso[TTo==e] for e in EVENTS}
        cond={}
        for e in EVENTS:
            cond[(e,"cl_run")]=tM[e][sM[e]>RUN_HI]
            cond[(e,"cl_rest")]=tM[e][sM[e]<REST_LO]
            cond[(e,"ol_run")]=tO[e][sO[e]>RUN_HI]
        recs=[]; PS={k:[] for k in cond}
        for u in vis:
            sp=sp_(u); area,layer=decode_layer(u_loc[u])
            row={"subject":subj,"area":area,"layer":layer,"uid":f"{subj}_{u}"}
            for (e,c),times in cond.items():
                row[f"{e}_{c}"]=np.mean(wrate(sp,times,RESP)-wrate(sp,times,BASE)) if len(times)>=3 else np.nan
                PS[(e,c)].append(psth(sp,times) if len(times)>=3 else np.full(len(TCEN),np.nan))
            recs.append(row)
        return pd.DataFrame(recs),{k:np.array(v) for k,v in PS.items()}
    finally: fh.close()

In [ ]:
# Run the batch (streams NWBs; ~30-230 s/session)
DFS=[]; PSs={}; t0=time.time()
for subj,aid in SESSIONS:
    d,ps=sm_extract(subj,aid); DFS.append(d); PSs[subj]=ps
    print(f"  {subj}: {len(d)} units ({time.time()-t0:.0f}s)")
SM=pd.concat(DFS,ignore_index=True)
print(f"POOLED: {len(SM)} units, {SM.subject.nunique()} sessions | areas: {SM.area.value_counts().to_dict()}")

## Contrast 1 — within-block deviance (all sessions)

Each deviant's baseline-subtracted response measures deviation from the ongoing standard flow
(the pre-event window *is* the standard). We split by running state because locomotion elevates
the baseline the deviant is measured against.

In [ ]:
def boot_ci_strat(vals,groups,n=5000,seed=0):
    rng=np.random.default_rng(seed); sess=np.unique(groups); meds=[]
    for _ in range(n):
        ii=np.concatenate([rng.choice(np.where(groups==s)[0],np.sum(groups==s),replace=True) for s in sess])
        meds.append(np.nanmedian(vals[ii]))
    return np.nanmedian(vals),np.percentile(meds,2.5),np.percentile(meds,97.5)

wb=[]
for e in ["motor_orientation_90","motor_orientation_45","motor_omission","motor_halt"]:
    for gate,cond in [("running","cl_run"),("rest","cl_rest")]:
        dd=SM.dropna(subset=[f"{e}_{cond}"]); v=dd[f"{e}_{cond}"].values
        if len(dd)<10: continue
        m,lo,hi=boot_ci_strat(v,dd.subject.values); p=ss.wilcoxon(v).pvalue
        wb.append(dict(event=e,gate=gate,n=len(dd),n_sess=dd.subject.nunique(),med=m,lo=lo,hi=hi,p=p,frac_pos=(v>0).mean()))
        print(f"  {e:22} {gate:8} n={len(dd):4}({dd.subject.nunique()}s) med={m:+.3f} [{lo:+.3f},{hi:+.3f}] p={p:.1e} {(v>0).mean():.0%}pos")
WB=pd.DataFrame(wb)

## Contrast 2 — closed-loop vs open-loop, and motor-dependence of the halt

The designed contrast (closed − open, running-matched) is computable only where both conditions
have running-coincident events. The `motor_halt` is the purely motor-contingent event; we also
ask whether its response depends on locomotion (running vs rest, within the closed loop).

In [ ]:
# running-coincidence yield per session (why the open-loop contrast is single-session)
def pw(rt,rv,times,win=(-1,0)):
    return np.array([np.nanmean(rv[(rt>=t+win[0])&(rt<t+win[1])]) if ((rt>=t+win[0])&(rt<t+win[1])).sum() else np.nan for t in times])
RY=[]
for subj,aid in SESSIONS:
    fh=h5py.File(remfile.File(s3_url("001637",aid)),"r")
    I=fh["intervals"]; gM=I["Sensory-motor mismatch block_presentations"]; gO=I["Control block 4_presentations"]
    rt,rv=get_running(fh); TTm=col(gM,"TrialType"); tsm=gM["start_time"][:]; TTo=col(gO,"TrialType"); tso=gO["start_time"][:]
    ht=tsm[TTm=="motor_halt"]; ho=tso[TTo=="motor_halt"]
    RY.append(dict(subject=subj,frac_run=float((rv>1).mean()),
                   cl_halt_run=int(np.nansum(pw(rt,rv,ht)>1)),ol_halt_run=int(np.nansum(pw(rt,rv,ho)>1))))
    fh.close()
RY=pd.DataFrame(RY).sort_values("frac_run",ascending=False).reset_index(drop=True)
print(RY.to_string(index=False))

# single powered session: closed - open, bootstrap CI (star only when CI excludes 0)
def boot_ci(paired,n=5000,seed=0):
    rng=np.random.default_rng(seed); meds=[np.nanmedian(rng.choice(paired,len(paired),replace=True)) for _ in range(n)]
    return np.nanmedian(paired),np.percentile(meds,2.5),np.percentile(meds,97.5)
POW=[s for s in RY.subject if RY.set_index("subject").loc[s,"cl_halt_run"]>=3 and RY.set_index("subject").loc[s,"ol_halt_run"]>=3]
print("\nsessions with both closed- and open-loop running halts:",POW)

### Figure — within-block deviance + halt motor-dependence

In [ ]:
TC=TCEN*1000; events=["motor_orientation_90","motor_orientation_45","motor_omission","motor_halt"]
elab=["orient 90°","orient 45°","omission","halt"]
def pool(cond): return np.vstack([PSs[s][cond] for s in PSs if cond in PSs[s]])
fig,axes=plt.subplots(1,3,figsize=(14,4.6)); fig.subplots_adjust(left=0.07,right=0.985,top=0.80,bottom=0.16,wspace=0.34)
# A within-block deviance forest
axA=axes[0]; yy=np.arange(len(events))
for gate,c,dy in [("running","#2c3e8c",-0.16),("rest","#e69f00",0.16)]:
    for i,e in enumerate(events):
        r=WB[(WB.event==e)&(WB.gate==gate)]
        if len(r)==0: continue
        r=r.iloc[0]; axA.plot([r.lo,r.hi],[yy[i]+dy]*2,color=c,lw=1.6,solid_capstyle="round")
        axA.plot(r.med,yy[i]+dy,"o",color=c,ms=5,label=gate if i==0 else "")
axA.axvline(0,color="k",lw=0.8,ls=":"); axA.set_yticks(yy); axA.set_yticklabels(elab,fontsize=8); axA.set_ylim(-0.6,3.6)
axA.set_xlabel("within-block deviance (Hz vs ongoing flow)"); axA.legend(frameon=False,fontsize=7,loc="center right",bbox_to_anchor=(1.0,0.62))
axA.set_title("A · Within-block deviance\nvisual deviants robust; halt is motor-specific",loc="left",fontsize=8.2)
# B halt per-session running vs rest
axB=axes[1]
for _,row in RY.iterrows():
    s=row.subject; d_=SM[SM.subject==s]; rr=d_["motor_halt_cl_run"].median(); rest=d_["motor_halt_cl_rest"].median()
    if pd.notna(rr) and pd.notna(rest):
        axB.plot([0,1],[rest,rr],"-",color="0.5",lw=1,zorder=1)
        axB.plot(0,rest,"o",color="#e69f00",ms=6,zorder=2); axB.plot(1,rr,"o",color="#2c3e8c",ms=6,zorder=2)
    elif pd.notna(rest): axB.plot(0,rest,"o",color="#e69f00",ms=6,mfc="none")
    elif pd.notna(rr): axB.plot(1,rr,"o",color="#2c3e8c",ms=6,mfc="none")
axB.axhline(0,color="k",lw=0.8,ls=":"); axB.set_xlim(-0.4,1.4); axB.set_xticks([0,1]); axB.set_xticklabels(["rest","running"],fontsize=8)
axB.set_ylabel("halt response (Hz, baseline-sub)")
axB.set_title("B · Flow-halt: motor-dependence\nnegative at rest (drive loss); pushed up by running",loc="left",fontsize=8.2)
# C halt PSTH running vs rest
axC=axes[2]
for cond,c,lab in [(("motor_halt","cl_run"),"#2c3e8c","running"),(("motor_halt","cl_rest"),"#e69f00","rest")]:
    M=pool(cond); m=np.nanmean(M,0); sem=np.nanstd(M,0)/np.sqrt(np.sum(~np.isnan(M[:,0])))
    ms=gaussian_filter1d(m,0.6); axC.plot(TC,ms,color=c,lw=1.8,label=lab); axC.fill_between(TC,ms-sem,ms+sem,color=c,alpha=0.2,lw=0)
axC.axvline(0,color="k",lw=0.5,ls=":"); axC.axvspan(0,300,color="#fdf0d5",zorder=0)
axC.set_xlim(-300,700); axC.set_xlabel("time from halt onset (ms)"); axC.set_ylabel("firing rate (Hz, baseline-sub)")
axC.legend(frameon=False,fontsize=7,loc="upper right",title="closed-loop",title_fontsize=6.5)
axC.set_title("C · Flow-halt PSTH, running vs rest\nlocomotion adds a positive component at halt",loc="left",fontsize=8.2)
fig.suptitle("Sensorimotor mismatch, within closed-loop block — visual deviants robust; flow-halt is motor-dependent",fontsize=9.4,y=0.955)
plt.show()

### Takeaway

- **Deviance detection operates in the sensorimotor block** (Contrast 1, all sessions): visual and
  omission deviants robustly exceed the ongoing standard flow (orient-90° up to +2.7 Hz,
  p ≈ 10⁻¹⁰¹). This is a population result on >1000 units.
- **The flow-halt is the motor-specific event.** Unlike the visual deviants, halting the flow
  *reduces* firing at rest (loss of visual drive) but is pushed up toward positive during running —
  a locomotion-dependent component consistent with a motor-based prediction error. Directionally
  robust, but underpowered (few sessions carry both running and rest halts).
- **The fully-designed closed-vs-open contrast is power-limited.** The open-loop control has only 8
  events/type and running rarely coincides, so a clean closed − open comparison exists in only one
  session (830794), where **omission** significantly exceeds playback (CI excludes 0) and halt
  points the same way (CI crosses 0). The motor-specific claim awaits data with more locomotion.

The sensorimotor block therefore *confirms* deviance detection at the population level and is
*consistent with* a motor-based prediction error, without yet establishing the latter at population
scale — an honest boundary set by the released data, not the analysis.